# Determining the Role of Big Data Applications in Fraud Detection and Financial Security in Financial Institutions of Ireland

---

**Programme:** MSc | National College of Ireland  
**Dataset:** Kaggle Credit Card Fraud Detection Dataset  
**Models:** Logistic Regression, Decision Tree, Random Forest  
**Language:** Python 3 | Libraries: Scikit-learn, Pandas, NumPy, Matplotlib, Seaborn

---

## Table of Contents
1. [Introduction & Research Objectives](#1-introduction)
2. [Environment Setup & Library Imports](#2-setup)
3. [Data Loading & Initial Exploration](#3-data-loading)
4. [Exploratory Data Analysis (EDA)](#4-eda)
5. [Data Preprocessing](#5-preprocessing)
6. [Model Implementation](#6-models)
7. [Model Evaluation & Comparison](#7-evaluation)
8. [Feature Importance Analysis](#8-features)
9. [Results Summary & Discussion](#9-results)
10. [Conclusion & Recommendations](#10-conclusion)

## 1. Introduction & Research Objectives <a id='1-introduction'></a>

### Research Background
Financial fraud has become a significant challenge for Irish financial institutions, with the Central Bank of Ireland reporting losses exceeding **€160 million** due to fraudulent payments in 2024. Traditional rule-based fraud detection systems struggle to identify complex and evolving fraud patterns, necessitating the adoption of advanced analytical approaches.

### Research Aim
To evaluate the role of Big Data applications in fraud detection and the improvement of financial security frameworks within financial institutions in Ireland.

### Research Objectives
1. To identify the applications of big data in fraud detection and financial security improvement
2. To examine the role of Big Data applications in improving regulatory standards and ethical considerations
3. To analyse the current challenges within financial institutions in Ireland for fraud detection
4. To provide recommendations for implementing Big Data applications into fraud detection systems

### Methodology
This research uses an **experimental machine learning approach** with the publicly available **Kaggle Credit Card Fraud Detection Dataset**. Three supervised learning algorithms are implemented and compared:
- **Logistic Regression** — baseline linear classifier
- **Decision Tree** — interpretable tree-based classifier
- **Random Forest** — ensemble learning method

Models are evaluated using: Accuracy, Precision, Recall, F1 Score, ROC-AUC, and Precision-Recall AUC.

## 2. Environment Setup & Library Imports <a id='2-setup'></a>

In [ ]:
# ── Standard Libraries ──────────────────────────────────────────────
import numpy as np
import pandas as pd
import time
import warnings
warnings.filterwarnings('ignore')

# ── Visualization ───────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

%matplotlib inline
plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'figure.facecolor': 'white',
})
sns.set_style('whitegrid')

# ── Machine Learning ────────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc,
    precision_recall_curve, average_precision_score
)

# ── Constants ────────────────────────────────────────────────────────
RANDOM_STATE = 42
TEST_SIZE = 0.3

MODEL_COLORS = {
    'Logistic Regression': '#2196F3',
    'Decision Tree': '#FF9800',
    'Random Forest': '#4CAF50',
}
CLASS_COLORS = {'Legitimate': '#4CAF50', 'Fraudulent': '#F44336'}

print('✅ All libraries imported successfully')
print(f'   NumPy:        {np.__version__}')
print(f'   Pandas:       {pd.__version__}')
print(f'   Scikit-learn: {__import__("sklearn").__version__}')

## 3. Data Loading & Initial Exploration <a id='3-data-loading'></a>

The dataset used is the **Credit Card Fraud Detection Dataset** from Kaggle, originally collected by the Machine Learning Group at ULB (Université Libre de Bruxelles). It contains **284,807 European credit card transactions** made over two days in September 2013.

**Features:**
- **V1–V28:** PCA-transformed numerical features (anonymised for confidentiality)
- **Time:** Seconds elapsed between each transaction and the first transaction
- **Amount:** Transaction amount in euros
- **Class:** Target variable (0 = Legitimate, 1 = Fraudulent)

In [ ]:
# Load the dataset
df = pd.read_csv('../data/creditcard.csv')
print(f'Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'\nColumn names: {list(df.columns)}')
df.head()

In [ ]:
# Dataset information
print('='*60)
print('DATASET INFORMATION')
print('='*60)
print(f'\nShape: {df.shape}')
print(f'\nData Types:')
print(df.dtypes.value_counts())
print(f'\nMemory Usage: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB')
print(f'\nMissing Values per Column:')
print(df.isnull().sum().sum(), '(total)')
print(f'\nDuplicate Rows: {df.duplicated().sum():,}')

In [ ]:
# Descriptive statistics
df.describe().round(4)

In [ ]:
# Class distribution
class_dist = df['Class'].value_counts()
class_pct = df['Class'].value_counts(normalize=True) * 100

print('='*60)
print('CLASS DISTRIBUTION')
print('='*60)
print(f'\nLegitimate (0): {class_dist[0]:>10,} transactions ({class_pct[0]:.3f}%)')
print(f'Fraudulent (1): {class_dist[1]:>10,} transactions ({class_pct[1]:.3f}%)')
print(f'\nImbalance Ratio: 1 fraud per {class_dist[0]//class_dist[1]} legitimate transactions')
print(f'\nThis is a HIGHLY IMBALANCED dataset — only {class_pct[1]:.3f}% of transactions are fraudulent.')
print('Standard accuracy alone is insufficient; Precision, Recall, and F1 Score are critical metrics.')

## 4. Exploratory Data Analysis (EDA) <a id='4-eda'></a>

Exploratory Data Analysis is conducted to understand the structure, distribution, and relationships within the dataset before model training.

### 4.1 Class Distribution Visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Bar chart
labels = ['Legitimate', 'Fraudulent']
colors = [CLASS_COLORS['Legitimate'], CLASS_COLORS['Fraudulent']]
bars = axes[0].bar(labels, class_dist.values, color=colors, edgecolor='black', linewidth=0.5)
for bar, val in zip(bars, class_dist.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1000,
                 f'{val:,}', ha='center', va='bottom', fontweight='bold')
axes[0].set_title('Transaction Count by Class', fontweight='bold')
axes[0].set_ylabel('Number of Transactions')

# Pie chart
axes[1].pie(class_dist.values, labels=labels, colors=colors, autopct='%1.3f%%',
            startangle=90, explode=(0, 0.15), shadow=True,
            textprops={'fontsize': 11})
axes[1].set_title('Percentage Distribution', fontweight='bold')

fig.suptitle('Class Distribution: Legitimate vs Fraudulent Transactions',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../results/figures/01_class_distribution.png', bbox_inches='tight', dpi=300)
plt.show()

print('\n📊 The dataset is extremely imbalanced with only 0.173% fraudulent transactions.')
print('   This imbalance is characteristic of real-world financial fraud datasets.')

### 4.2 Transaction Amount Analysis

In [ ]:
print('='*60)
print('TRANSACTION AMOUNT STATISTICS')
print('='*60)

for label, cls in [('Legitimate', 0), ('Fraudulent', 1)]:
    subset = df[df['Class'] == cls]['Amount']
    print(f'\n{label} Transactions:')
    print(f'  Count:  {len(subset):,}')
    print(f'  Mean:   €{subset.mean():.2f}')
    print(f'  Median: €{subset.median():.2f}')
    print(f'  Std:    €{subset.std():.2f}')
    print(f'  Min:    €{subset.min():.2f}')
    print(f'  Max:    €{subset.max():.2f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for cls, label in [(0, 'Legitimate'), (1, 'Fraudulent')]:
    subset = df[df['Class'] == cls]['Amount']
    color = CLASS_COLORS[label]
    axes[0].hist(subset, bins=50, alpha=0.7, label=label, color=color, edgecolor='black', linewidth=0.3)
    axes[1].hist(subset, bins=50, alpha=0.7, label=label, color=color, edgecolor='black', linewidth=0.3)

axes[0].set_title('Transaction Amount Distribution', fontweight='bold')
axes[0].set_xlabel('Amount (€)')
axes[0].set_ylabel('Frequency')
axes[0].legend()

axes[1].set_title('Transaction Amount Distribution (Log Scale)', fontweight='bold')
axes[1].set_xlabel('Amount (€)')
axes[1].set_ylabel('Frequency (log)')
axes[1].set_yscale('log')
axes[1].legend()

fig.suptitle('Transaction Amount Distribution by Class', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../results/figures/02_amount_distribution.png', bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

data_box = [df[df['Class']==0]['Amount'], df[df['Class']==1]['Amount']]

for i, (ax, yscale) in enumerate(zip(axes, ['linear', 'log'])):
    bp = ax.boxplot(data_box, labels=['Legitimate', 'Fraudulent'],
                    patch_artist=True, notch=True)
    bp['boxes'][0].set_facecolor(CLASS_COLORS['Legitimate'])
    bp['boxes'][1].set_facecolor(CLASS_COLORS['Fraudulent'])
    title = 'Transaction Amount by Class'
    if yscale == 'log':
        ax.set_yscale('log')
        title += ' (Log Scale)'
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Amount (€)')

fig.suptitle('Transaction Amount Comparison', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../results/figures/05_amount_boxplot.png', bbox_inches='tight', dpi=300)
plt.show()

### 4.3 Temporal Distribution of Transactions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['Time']/3600, bins=48, color='#2196F3', alpha=0.7,
             edgecolor='black', linewidth=0.3)
axes[0].set_title('All Transactions Over Time', fontweight='bold')
axes[0].set_xlabel('Time (hours)')
axes[0].set_ylabel('Number of Transactions')

for cls, label in [(0, 'Legitimate'), (1, 'Fraudulent')]:
    subset = df[df['Class']==cls]['Time']/3600
    axes[1].hist(subset, bins=48, alpha=0.6, label=label,
                 color=CLASS_COLORS[label], edgecolor='black', linewidth=0.3)
axes[1].set_title('Transactions Over Time by Class', fontweight='bold')
axes[1].set_xlabel('Time (hours)')
axes[1].set_ylabel('Frequency')
axes[1].legend()

fig.suptitle('Temporal Distribution of Transactions', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../results/figures/03_time_distribution.png', bbox_inches='tight', dpi=300)
plt.show()

print('📊 The dataset covers approximately 48 hours of transactions.')
print('   Transaction volume shows a cyclical pattern, likely corresponding to day/night cycles.')

### 4.4 Feature Correlation Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(16, 14))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='RdBu_r', center=0,
            annot=False, linewidths=0.5, ax=ax,
            cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/figures/04_correlation_heatmap.png', bbox_inches='tight', dpi=300)
plt.show()

# Show features most correlated with fraud
print('\nFeatures most correlated with Fraud (Class):')
fraud_corr = corr['Class'].drop('Class').abs().sort_values(ascending=False)
for feat, val in fraud_corr.head(10).items():
    direction = '+' if corr.loc[feat, 'Class'] > 0 else '-'
    print(f'  {feat:>8s}: {direction}{val:.4f}')

### 4.5 Key Feature Distributions by Class

In [ ]:
top_features = ['V1', 'V2', 'V3', 'V4', 'V10', 'V11', 'V12', 'V14', 'V16', 'V17']

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for i, feat in enumerate(top_features):
    for cls, label in [(0, 'Legitimate'), (1, 'Fraudulent')]:
        subset = df[df['Class']==cls][feat]
        axes[i].hist(subset, bins=50, alpha=0.6, label=label,
                     color=CLASS_COLORS[label], density=True)
    axes[i].set_title(feat, fontweight='bold')
    if i == 0:
        axes[i].legend(fontsize=8)

fig.suptitle('Distribution of Key PCA Features by Class',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../results/figures/06_feature_distributions.png', bbox_inches='tight', dpi=300)
plt.show()

print('📊 Features V14, V12, V10, and V17 show the strongest separation between classes.')
print('   These features are expected to be the most important for fraud detection.')

## 5. Data Preprocessing <a id='5-preprocessing'></a>

The preprocessing steps include:
1. **Duplicate removal** — removing any repeated records
2. **Missing value check** — verifying data completeness
3. **Feature scaling** — standardising Time and Amount (V1–V28 are already PCA-transformed)
4. **Train/test split** — 70/30 stratified split to preserve class distribution

In [ ]:
# ── Step 1: Remove duplicates ───────────────────────────────────────
print('STEP 1: Checking for duplicates')
initial = len(df)
df_clean = df.drop_duplicates()
removed = initial - len(df_clean)
print(f'  Removed {removed:,} duplicate rows')
print(f'  Remaining: {len(df_clean):,} rows')

# ── Step 2: Missing values ──────────────────────────────────────────
print('\nSTEP 2: Checking for missing values')
missing = df_clean.isnull().sum().sum()
print(f'  Total missing values: {missing}')

# ── Step 3: Feature scaling ─────────────────────────────────────────
print('\nSTEP 3: Feature scaling (StandardScaler on Time and Amount)')
scaler = StandardScaler()
df_clean['Time_scaled'] = scaler.fit_transform(df_clean[['Time']])
df_clean['Amount_scaled'] = scaler.fit_transform(df_clean[['Amount']])
df_processed = df_clean.drop(['Time', 'Amount'], axis=1)
print(f'  Scaled Time → Time_scaled')
print(f'  Scaled Amount → Amount_scaled')
print(f'  Final feature count: {df_processed.shape[1] - 1} (excluding target)')

# ── Step 4: Train/Test split ────────────────────────────────────────
print('\nSTEP 4: Splitting data (70% train / 30% test, stratified)')
X = df_processed.drop('Class', axis=1)
y = df_processed['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

print(f'\n  Training set: {X_train.shape[0]:,} samples')
print(f'    Legitimate: {(y_train==0).sum():,} | Fraudulent: {(y_train==1).sum():,}')
print(f'  Testing set:  {X_test.shape[0]:,} samples')
print(f'    Legitimate: {(y_test==0).sum():,} | Fraudulent: {(y_test==1).sum():,}')
print(f'\n✅ Data preprocessing complete!')

## 6. Model Implementation <a id='6-models'></a>

Three supervised classification algorithms are implemented using Scikit-learn:

| Model | Type | Key Characteristics |
|-------|------|--------------------|
| **Logistic Regression** | Linear | Fast, interpretable, baseline model |
| **Decision Tree** | Tree-based | Explainable rules, prone to overfitting |
| **Random Forest** | Ensemble | Multiple trees combined, robust performance |

All models use `class_weight='balanced'` to handle the class imbalance by assigning higher weights to the minority (fraud) class.

In [ ]:
# ── Define Models ──────────────────────────────────────────────────
models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE,
        solver='lbfgs',
        class_weight='balanced'
    ),
    'Decision Tree': DecisionTreeClassifier(
        random_state=RANDOM_STATE,
        max_depth=10,
        min_samples_split=5,
        min_samples_leaf=2,
        class_weight='balanced'
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=100,
        random_state=RANDOM_STATE,
        max_depth=15,
        min_samples_split=5,
        min_samples_leaf=2,
        class_weight='balanced',
        n_jobs=-1
    ),
}

# ── Train All Models ────────────────────────────────────────────────
results = {}

print('='*60)
print('MODEL TRAINING')
print('='*60)

for name, model in models.items():
    print(f'\n▶ Training {name}...')
    start = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start
    
    # Predictions
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc_val = auc(fpr, tpr)
    pr_prec, pr_rec, _ = precision_recall_curve(y_test, y_proba)
    pr_auc_val = average_precision_score(y_test, y_proba)
    
    results[name] = {
        'model': model, 'y_pred': y_pred, 'y_proba': y_proba,
        'accuracy': acc, 'precision': prec, 'recall': rec,
        'f1_score': f1, 'confusion_matrix': cm,
        'fpr': fpr, 'tpr': tpr, 'roc_auc': roc_auc_val,
        'pr_precision': pr_prec, 'pr_recall': pr_rec, 'pr_auc': pr_auc_val,
        'training_time': train_time,
        'report': classification_report(y_test, y_pred, output_dict=True)
    }
    
    print(f'  ✓ Trained in {train_time:.2f}s')
    print(f'  Accuracy:  {acc:.4f}  |  Precision: {prec:.4f}')
    print(f'  Recall:    {rec:.4f}  |  F1 Score:  {f1:.4f}')
    print(f'  ROC AUC:   {roc_auc_val:.4f}  |  PR AUC:    {pr_auc_val:.4f}')
    print(f'  Confusion Matrix: TN={cm[0][0]:,} FP={cm[0][1]:,} FN={cm[1][0]:,} TP={cm[1][1]:,}')

print('\n✅ All models trained successfully!')

## 7. Model Evaluation & Comparison <a id='7-evaluation'></a>

### 7.1 Model Performance Comparison Table

In [ ]:
# Build comparison table
comparison_data = []
for name, res in results.items():
    comparison_data.append({
        'Model': name,
        'Accuracy': f"{res['accuracy']:.4f}",
        'Precision': f"{res['precision']:.4f}",
        'Recall': f"{res['recall']:.4f}",
        'F1 Score': f"{res['f1_score']:.4f}",
        'ROC AUC': f"{res['roc_auc']:.4f}",
        'PR AUC': f"{res['pr_auc']:.4f}",
        'Training Time': f"{res['training_time']:.2f}s"
    })

comparison_df = pd.DataFrame(comparison_data)
print('MODEL PERFORMANCE COMPARISON')
print('='*90)
display(comparison_df.style.set_properties(**{'text-align': 'center'}))

### 7.2 Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, (name, res) in enumerate(results.items()):
    cm = res['confusion_matrix']
    sns.heatmap(cm, annot=True, fmt=',d', cmap='Blues', ax=axes[i],
                xticklabels=['Legitimate', 'Fraud'],
                yticklabels=['Legitimate', 'Fraud'],
                linewidths=1, linecolor='black')
    axes[i].set_title(f"{name}\nAccuracy: {res['accuracy']:.4f}", fontweight='bold')
    axes[i].set_ylabel('Actual')
    axes[i].set_xlabel('Predicted')

fig.suptitle('Confusion Matrices — Model Comparison', fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout()
plt.savefig('../results/figures/07_confusion_matrices.png', bbox_inches='tight', dpi=300)
plt.show()

print('\n📊 Analysis:')
for name, res in results.items():
    cm = res['confusion_matrix']
    print(f'  {name}:')
    print(f'    • False Positives: {cm[0][1]:,} (legitimate flagged as fraud)')
    print(f'    • False Negatives: {cm[1][0]:,} (fraud missed)')

### 7.3 ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

for name, res in results.items():
    ax.plot(res['fpr'], res['tpr'],
            label=f"{name} (AUC = {res['roc_auc']:.4f})",
            color=MODEL_COLORS[name], linewidth=2)

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier (AUC = 0.5000)')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('Receiver Operating Characteristic (ROC) Curves', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/figures/08_roc_curves.png', bbox_inches='tight', dpi=300)
plt.show()

print('📊 ROC AUC scores indicate how well each model separates fraud from legitimate transactions.')
print('   AUC = 1.0 is perfect; AUC = 0.5 is random guessing.')

### 7.4 Precision-Recall Curves

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

for name, res in results.items():
    ax.plot(res['pr_recall'], res['pr_precision'],
            label=f"{name} (AP = {res['pr_auc']:.4f})",
            color=MODEL_COLORS[name], linewidth=2)

ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Precision-Recall Curves', fontsize=14, fontweight='bold')
ax.legend(loc='lower left', fontsize=11)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/figures/09_precision_recall_curves.png', bbox_inches='tight', dpi=300)
plt.show()

print('📊 Precision-Recall curves are more informative than ROC for imbalanced datasets.')
print('   Higher PR AUC means better performance on the minority (fraud) class.')

### 7.5 Model Metrics Comparison (Bar Chart)

In [ ]:
metrics_keys = ['accuracy', 'precision', 'recall', 'f1_score', 'roc_auc']
metric_labels = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC AUC']
model_names = list(results.keys())

x = np.arange(len(metric_labels))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 7))

for i, name in enumerate(model_names):
    values = [results[name][k] for k in metrics_keys]
    bars = ax.bar(x + i*width, values, width, label=name,
                  color=MODEL_COLORS[name], edgecolor='black', linewidth=0.5)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.008,
                f'{val:.3f}', ha='center', va='bottom', fontsize=7.5, fontweight='bold')

ax.set_xlabel('Metric', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels(metric_labels)
ax.legend(fontsize=11)
ax.set_ylim(0, 1.15)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../results/figures/10_metrics_comparison.png', bbox_inches='tight', dpi=300)
plt.show()

### 7.6 Training Time Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

names = list(results.keys())
times = [results[n]['training_time'] for n in names]
colors = [MODEL_COLORS[n] for n in names]

bars = ax.bar(names, times, color=colors, edgecolor='black', linewidth=0.5)
for bar, t in zip(bars, times):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{t:.2f}s', ha='center', va='bottom', fontweight='bold')

ax.set_ylabel('Training Time (seconds)')
ax.set_title('Model Training Time Comparison', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../results/figures/11_training_time.png', bbox_inches='tight', dpi=300)
plt.show()

### 7.7 Detailed Classification Reports

In [ ]:
for name, res in results.items():
    print(f'\n{"="*60}')
    print(f'CLASSIFICATION REPORT: {name}')
    print('='*60)
    print(classification_report(y_test, res['y_pred'],
          target_names=['Legitimate', 'Fraudulent']))

## 8. Feature Importance Analysis <a id='8-features'></a>

Understanding which features contribute most to fraud detection helps financial institutions focus monitoring efforts.

In [ ]:
feature_names = list(X_train.columns)

# Decision Tree and Random Forest feature importances
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

for idx, name in enumerate(['Decision Tree', 'Random Forest']):
    model = results[name]['model']
    imp = pd.DataFrame({'Feature': feature_names, 'Importance': model.feature_importances_})
    imp = imp.sort_values('Importance', ascending=False).head(15)
    
    axes[idx].barh(imp['Feature'], imp['Importance'],
                   color=MODEL_COLORS[name], edgecolor='black', linewidth=0.3)
    axes[idx].set_title(f'{name} — Top 15 Features', fontweight='bold')
    axes[idx].set_xlabel('Importance')
    axes[idx].invert_yaxis()

fig.suptitle('Feature Importance Analysis', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../results/figures/12_feature_importance.png', bbox_inches='tight', dpi=300)
plt.show()

# Print top features
print('\nTop 10 Most Important Features by Model:')
for name in ['Decision Tree', 'Random Forest']:
    model = results[name]['model']
    imp = pd.DataFrame({'Feature': feature_names, 'Importance': model.feature_importances_})
    imp = imp.sort_values('Importance', ascending=False).head(10)
    print(f'\n  {name}:')
    for _, row in imp.iterrows():
        print(f'    {row["Feature"]:>15s}: {row["Importance"]:.6f}')

In [ ]:
# Logistic Regression coefficients
lr_model = results['Logistic Regression']['model']
coef_df = pd.DataFrame({'Feature': feature_names, 'Coefficient': lr_model.coef_[0]})
coef_df['Abs_Coefficient'] = coef_df['Coefficient'].abs()
coef_df = coef_df.sort_values('Abs_Coefficient', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#F44336' if c < 0 else '#4CAF50' for c in coef_df['Coefficient']]
ax.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors,
        edgecolor='black', linewidth=0.3)
ax.set_title('Logistic Regression — Top 15 Feature Coefficients',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Coefficient Value')
ax.invert_yaxis()
ax.axvline(x=0, color='black', linewidth=0.8)
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('../results/figures/13_lr_coefficients.png', bbox_inches='tight', dpi=300)
plt.show()

print('📊 Positive coefficients increase fraud probability; negative decrease it.')
print('   V14 and Amount_scaled are the most influential features across all models.')

## 9. Results Summary & Discussion <a id='9-results'></a>

In [ ]:
print('╔' + '═'*70 + '╗')
print('║            RESULTS SUMMARY & DISCUSSION                              ║')
print('╚' + '═'*70 + '╝')

print(f'''
DATASET OVERVIEW:
  • 284,807 credit card transactions (European cardholders, September 2013)
  • Highly imbalanced: 99.827% legitimate, 0.173% fraudulent (492 cases)
  • 30 features: V1–V28 (PCA), Time, and Amount
  • After cleaning: 283,726 rows (1,081 duplicates removed)

MODEL COMPARISON RESULTS:
{'='*70}''')

# Comparison table
display(comparison_df)

# Best model identification
best_name = max(results, key=lambda k: results[k]['f1_score'])
best = results[best_name]

print(f'''
{'='*70}
BEST MODEL: {best_name}
{'='*70}
  F1 Score:  {best['f1_score']:.4f}   (best balance of precision and recall)
  Precision: {best['precision']:.4f}   (of flagged transactions, {best['precision']*100:.1f}% are real fraud)
  Recall:    {best['recall']:.4f}   (detects {best['recall']*100:.1f}% of all fraud)
  ROC AUC:   {best['roc_auc']:.4f}
  PR AUC:    {best['pr_auc']:.4f}

KEY FINDINGS:

1. RANDOM FOREST outperforms both Logistic Regression and Decision Tree
   across all key metrics, achieving the highest F1 Score ({best['f1_score']:.4f})
   and the best precision-recall trade-off.

2. LOGISTIC REGRESSION achieves the highest recall ({results['Logistic Regression']['recall']:.4f}),
   detecting {results['Logistic Regression']['recall']*100:.1f}% of fraud cases, but at the cost of very
   low precision ({results['Logistic Regression']['precision']:.4f}) — generating many false positives.

3. DECISION TREE provides moderate performance but shows room for
   improvement in both precision and recall compared to Random Forest.

4. FEATURE V14 is the most important predictor of fraud across all three models,
   followed by V10, V17, V4, and V12.

5. CLASS IMBALANCE (577:1 ratio) is a critical challenge. Using
   class_weight="balanced" helps but the extreme imbalance still
   affects precision for simpler models.
''')

## 10. Conclusion & Recommendations <a id='10-conclusion'></a>

In [ ]:
print('''
╔══════════════════════════════════════════════════════════════════════╗
║                    CONCLUSION & RECOMMENDATIONS                     ║
╚══════════════════════════════════════════════════════════════════════╝

CONCLUSION:
───────────
This research has demonstrated the effectiveness of Big Data applications
and machine learning models in detecting fraudulent financial transactions.
The comparative analysis of Logistic Regression, Decision Tree, and Random
Forest on the Credit Card Fraud Detection Dataset reveals that:

  • Random Forest is the most effective algorithm for fraud detection,
    providing the best balance between precision and recall through
    ensemble learning.

  • Big Data analytics, combined with machine learning, can process
    large volumes of financial transactions and identify complex
    patterns of fraudulent behaviour that traditional rule-based
    systems cannot detect.

  • The extreme class imbalance in real-world fraud datasets is a
    fundamental challenge that requires specialised handling through
    techniques such as class weighting, sampling strategies, or
    cost-sensitive learning.

RECOMMENDATIONS FOR IRISH FINANCIAL INSTITUTIONS:
──────────────────────────────────────────────────
  1. ADOPT ENSEMBLE METHODS: Implement Random Forest or similar
     ensemble algorithms as the primary fraud detection engine,
     rather than single classifiers.

  2. REAL-TIME PROCESSING: Deploy Big Data frameworks (e.g., Apache
     Spark, Kafka) to enable real-time transaction scoring at scale.

  3. CONTINUOUS MODEL RETRAINING: Fraud patterns evolve — models
     should be retrained periodically with fresh data to maintain
     detection accuracy.

  4. MULTI-LAYERED DEFENCE: Combine ML-based detection with rule-based
     systems, behavioural analytics, and human review for a robust
     fraud prevention strategy.

  5. REGULATORY COMPLIANCE: Ensure fraud detection systems comply
     with GDPR, the Irish Data Protection Act 2018, and the EU AI
     Act — particularly around transparency, fairness, and data
     protection.

  6. EXPLAINABILITY: Augment Random Forest with model-agnostic
     explainability tools (SHAP, LIME) so flagged transactions can
     be reviewed and justified by compliance teams.

  7. ADDRESS CLASS IMBALANCE: Implement SMOTE or other oversampling
     techniques alongside cost-sensitive learning to improve
     detection of rare fraud cases.

FUTURE WORK:
────────────
  • Evaluate deep learning approaches (e.g., autoencoders, LSTMs)
    for anomaly-based fraud detection.
  • Test on larger, more recent Irish financial transaction datasets.
  • Incorporate SMOTE and other resampling techniques.
  • Explore hyperparameter optimisation via grid/random search.
  • Investigate federated learning for privacy-preserving fraud
    detection across institutions.
''')

In [ ]:
# Save all results
import os
os.makedirs('../results/tables', exist_ok=True)
os.makedirs('../results/models', exist_ok=True)

# Save comparison table
comparison_df.to_csv('../results/tables/model_comparison.csv', index=False)

# Save classification reports
for name, res in results.items():
    report_df = pd.DataFrame(res['report']).transpose()
    fname = name.lower().replace(' ', '_') + '_report.csv'
    report_df.to_csv(f'../results/tables/{fname}')

# Save models
import joblib
for name, res in results.items():
    fname = name.lower().replace(' ', '_') + '.joblib'
    joblib.dump(res['model'], f'../results/models/{fname}')

print('✅ All results, tables, and models saved successfully!')
print(f'   Figures: ../results/figures/ ({len(os.listdir("../results/figures"))} files)')
print(f'   Tables:  ../results/tables/ ({len(os.listdir("../results/tables"))} files)')
print(f'   Models:  ../results/models/ ({len(os.listdir("../results/models"))} files)')